# FA2: Cucumber Leaf Disease Classification using MobileNetV2
## Transfer Learning for Image Classification
**Student:** [Your Name]  
**PRN:** [Your PRN]

## 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelBinarizer

import cv2
from tqdm import tqdm

## 2. Data Loading and Preprocessing

In [ ]:
# Set paths
TRAIN_DIR = 'data/train'
VAL_DIR = 'data/validation'
TEST_DIR = 'data/test'

# Image dimensions
IMG_WIDTH = 224
IMG_HEIGHT = 224
BATCH_SIZE = 32

# Classes
CLASSES = ['Downy Mildew', 'Healthy', 'Powdery Mildew']

# ImageNet normalization
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

# No augmentation for validation/test
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load training data
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Load validation data
val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Load test data
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('Training samples:', train_generator.samples)
print('Validation samples:', val_generator.samples)
print('Test samples:', test_generator.samples)

## 3. Build MobileNetV2 Model

In [ ]:
# Load pre-trained MobileNetV2
base_model = MobileNetV2(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
                         include_top=False,
                         weights='imagenet')

# Freeze base model layers
base_model.trainable = False

# Build custom model
model = Model(inputs=base_model.input, outputs=base_model.output)
model.add(GlobalAveragePooling2D())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(3, activation='softmax'))

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. Train the Model

In [ ]:
# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001
)

# Train model
history = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=[early_stopping, reduce_lr]
)

## 5. Evaluate on Test Set

In [ ]:
# Predictions
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

# Accuracy
test_accuracy = np.mean(y_pred_classes == y_true)
print(f'Test Accuracy: {test_accuracy:.4f}')

# Classification Report
print('\nClassification Report:')
print(classification_report(y_true, y_pred_classes, target_names=CLASSES))

## 6. Visualize Results

In [ ]:
# Plot accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('outputs/MobileNet/MobileNet_accuracy_loss_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix - MobileNetV2')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('outputs/MobileNet/MobileNet_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Save Model and Results

In [ ]:
# Save model
model.save('models/MobileNetV2_FA2.h5')

# Save model architecture
with open('models/MobileNetV2_FA2.txt', 'w') as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))

# Save classification report
report = classification_report(y_true, y_pred_classes, target_names=CLASSES)
with open('outputs/MobileNet/MobileNet_classification_report.txt', 'w') as f:
    f.write('MobileNetV2 Classification Report\n')
    f.write('='*50 + '\n')
    f.write(f'Test Accuracy: {test_accuracy:.4f}\n\n')
    f.write(report)

print('Model and results saved!')